# 7. Prédictions et visualisation des détections (`detect_dual.py`)

Ce notebook utilise le modèle entraîné (`best.pt`) pour prédire et visualiser les véhicules sur les images du dossier `test` avec un seuil de confiance `conf-thres = 0.25`, puis analyse les détections correctes, les objets manqués et les fausses détections en lien avec les métriques de l'étape 6.

In [ ]:
from pathlib import Path
import subprocess
import sys
import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

YOLO_DIR = Path("../yolov9").resolve()
DATA_DIR = Path("../trafic_data").resolve()
DATA_YAML = DATA_DIR / "data_1.yaml"
TEST_IMAGES_DIR = DATA_DIR / "test/images"
TEST_LABELS_DIR = DATA_DIR / "test/labels"
DETECT_SCRIPT = YOLO_DIR / "detect_dual.py"
BEST_PT = YOLO_DIR / "runs/train/yolov9c_vehicles_e10/weights/best.pt"
PRED_NAME = "yolov9c_vehicles_test_pred"
PRED_DIR = YOLO_DIR / "runs/detect" / PRED_NAME

HAS_CUDA = torch.cuda.is_available()
DEVICE = "0" if HAS_CUDA else "cpu"
print(f"Device détecté : {DEVICE} | Images test : {len(list(TEST_IMAGES_DIR.glob('*')))} | best.pt existe={BEST_PT.exists()}")

detect_cmd = [
    sys.executable,
    str(DETECT_SCRIPT),
    "--weights", str(BEST_PT),
    "--source", str(TEST_IMAGES_DIR),
    "--data", str(DATA_YAML),
    "--img", "640",
    "--conf-thres", "0.25",
    "--iou-thres", "0.45",
    "--device", DEVICE,
    "--save-txt",
    "--save-conf",
    "--project", str(YOLO_DIR / "runs/detect"),
    "--name", PRED_NAME,
    "--exist-ok",
]
print("Commande de prédiction prête :\n", " ".join(detect_cmd))

In [ ]:
# Exécuter l'inférence et enregistrer les images annotées + labels prédits
assert BEST_PT.exists(), (
    f"Checkpoint introuvable : {BEST_PT}. "
    "Exécutez d'abord l'entraînement dans trainYolov9.ipynb (sur GPU) pour générer best.pt."
)

proc = subprocess.run(detect_cmd, cwd=str(YOLO_DIR), capture_output=True, text=True)
print(proc.stdout)
if proc.stderr:
    print(proc.stderr)
print("Code de retour detect_dual.py :", proc.returncode)

In [ ]:
# Afficher au moins 5 exemples d'images prédites avec boîtes englobantes et classes
EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}
pred_images = sorted([
    p for p in PRED_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in EXTENSIONS
]) if PRED_DIR.exists() else []

if len(pred_images) >= 5:
    exemples = pred_images[:6]
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    for ax, img_path in zip(axes.flatten(), exemples):
        with Image.open(img_path) as im:
            ax.imshow(im)
        ax.set_title(img_path.name, fontsize=9)
        ax.axis("off")
    plt.suptitle("Exemples de prédictions YOLOv9-C sur le jeu de test (conf >= 0.25)", fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print(f"Images prédites insuffisantes ({len(pred_images)} trouvées dans {PRED_DIR}). Lancez d'abord detect_dual.py.")

In [ ]:
# Analyse quantitative des erreurs visuelles (TP, FP, FN) par image à IoU >= 0.50
def load_boxes(txt_path, is_pred=False):
    if not txt_path.exists():
        return []
    boxes = []
    for line in txt_path.read_text(encoding="utf-8").splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        cls_id = int(float(parts[0]))
        xc, yc, w, h = map(float, parts[1:5])
        conf = float(parts[5]) if (is_pred and len(parts) >= 6) else 1.0
        boxes.append((cls_id, xc - w/2, yc - h/2, xc + w/2, yc + h/2, conf))
    return boxes

def compute_iou(b1, b2):
    ix1, iy1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    ix2, iy2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    area1 = max(0.0, b1[2] - b1[0]) * max(0.0, b1[3] - b1[1])
    area2 = max(0.0, b2[2] - b2[0]) * max(0.0, b2[3] - b2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0.0

pred_lbl_dir = PRED_DIR / "labels"
if pred_lbl_dir.exists():
    total_tp = total_fp = total_fn = 0
    for gt_file in sorted(TEST_LABELS_DIR.glob("*.txt")):
        gt = load_boxes(gt_file, is_pred=False)
        pr = sorted(load_boxes(pred_lbl_dir / gt_file.name, is_pred=True), key=lambda x: -x[5])
        matched_gt = set()
        tp = fp = 0
        for p in pr:
            best_iou, best_j = 0.0, -1
            for j, g in enumerate(gt):
                if j in matched_gt or g[0] != p[0]:
                    continue
                iou = compute_iou(p[1:5], g[1:5])
                if iou > best_iou:
                    best_iou, best_j = iou, j
            if best_iou >= 0.50 and best_j != -1:
                matched_gt.add(best_j)
                tp += 1
            else:
                fp += 1
        fn = len(gt) - len(matched_gt)
        total_tp += tp
        total_fp += fp
        total_fn += fn
    print(f"Bilan visuel à conf>=0.25, IoU>=0.50 -> TP (correctes): {total_tp} | FP (fausses détections): {total_fp} | FN (objets manqués): {total_fn}")
else:
    print(f"Labels prédits introuvables dans {pred_lbl_dir}. Exécutez d'abord detect_dual.py avec --save-txt.")

### Analyse qualitative et lien avec les métriques d'évaluation

1. **Grille d'analyse visuelle (`conf >= 0.25`) :**
   - **Détections correctes (True Positives) :** véhicules bien localisés et correctement classés (ex. `car`, `bus`, `rickshaw`, `motorbike`, `three wheelers -CNG-` bien visibles).
   - **Objets manqués (False Negatives) :** véhicules très petits/éloignés à l'arrière-plan, partiellement occultés dans le trafic dense, ou appartenant à des classes rares (`policecar`, `scooter`, `ambulance`).
   - **Fausses détections (False Positives) :** confusions entre catégories visuellement proches (ex. `car` vs `suv`/`taxi`, `minivan` vs `van`/`minibus`, `motorbike` vs `scooter`) ou doubles détections sur un même véhicule.

2. **Comparaison avec les métriques de l'étape 6 (`val_dual.py`) :**
   - Un **Recall** faible sur une classe se traduit visuellement par de nombreux **objets manqués (FN)**.
   - Une **Precision** faible se manifeste par des **fausses détections (FP)** ou des **confusions de classes** visibles sur la `confusion_matrix.png`.
   - Comme la visualisation filtre à `conf >= 0.25` (contre `0.001` en évaluation), certaines détections peu confiantes comptées dans `mAP@50` n'apparaissent pas sur les images finales : on privilégie la lisibilité visuelle (moins de FP) au détriment d'un peu de Recall.